In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Azure Storage Authentication Configuration ---
STORAGE_ACCOUNT = "metadataprojectdev"
# Define target paths pointing to your Cloud Container
CONTAINER_NAME = "clientalpha"
BASE_ABFSS_URL = f"abfss://{CONTAINER_NAME}@{STORAGE_ACCOUNT}.dfs.core.windows.net"



In [0]:
# Load local metadata contract file
metadata_path = f"{BASE_ABFSS_URL}/data/metadata.yaml"
metadata_text = "\n".join(row.value for row in spark.read.text(metadata_path).collect())
metadata = yaml.safe_load(metadata_text.removeprefix("```yaml\n").removesuffix("\n```"))


In [0]:

# 2.  DYNAMIC CLOUD CONTAINER READER

import fnmatch
import os

source_root = f"{BASE_ABFSS_URL}/data"
available_paths = [item.path for item in dbutils.fs.ls(source_root)]
format_priority = ["CSV", "TSV", "JSONL", "JSON", "XML", "PARQUET"]
extension_to_format = {
    ".csv": "CSV",
    ".tsv": "TSV",
    ".json": "JSON",
    ".jsonl": "JSONL",
    ".xml": "XML",
    ".parquet": "PARQUET",
}

selected_source = None
file_path = None
file_format = None
source_entries = metadata.get("sources") or [{
    "source_id": metadata.get("delivery_metadata", {}).get("delivery_id", "default_source"),
    "source_config": metadata["source_config"],
    "schema_definition": metadata["schema_definition"],
    "quality_rules": metadata.get("quality_rules", []),
    "destination_config": metadata["destination_config"],
}]

for source in source_entries:
    src_config = source["source_config"]
    pattern = src_config["file_pattern"]

    matched_candidates = []
    for path in available_paths:
        file_name = os.path.basename(path)
        matched_format = extension_to_format.get(os.path.splitext(file_name)[1].lower())
        if matched_format and fnmatch.fnmatch(file_name, pattern):
            matched_candidates.append((path, matched_format))

    if not matched_candidates:
        pattern_stem, _ = os.path.splitext(pattern)
        for path in available_paths:
            file_name = os.path.basename(path)
            file_stem, file_ext = os.path.splitext(file_name)
            matched_format = extension_to_format.get(file_ext.lower())
            if matched_format and file_stem == pattern_stem:
                matched_candidates.append((path, matched_format))

    if matched_candidates:
        matched_candidates.sort(key=lambda item: format_priority.index(item[1]))
        file_path, file_format = matched_candidates[0]
        selected_source = source
        break

if selected_source is None:
    print(f"[DEBUG] Available files: {[os.path.basename(p) for p in available_paths]}")
    print(f"[DEBUG] Expected pattern(s): {[s['source_config']['file_pattern'] for s in source_entries]}")
    raise FileNotFoundError(f"No supported ABFSS source files matched metadata patterns under {source_root}")

src_config = selected_source["source_config"]
print(f"[INFO] Selected source: {selected_source['source_id']}")
print(f"[INFO] Reading {file_format} file from Azure Cloud path: {file_path}")

if file_format in ("CSV", "TSV"):
    delimiter = "\t" if file_format == "TSV" else src_config.get("delimiter", ",")
    raw_df = spark.read.format("csv") \
        .option("header", src_config.get("header", True)) \
        .option("delimiter", delimiter) \
        .option("encoding", src_config.get("encoding", "UTF-8")) \
        .load(file_path)
elif file_format == "JSON":
    raw_df = spark.read.option("multiline", "true").json(file_path)
elif file_format == "JSONL":
    raw_df = spark.read.json(file_path)
elif file_format == "XML":
    raw_df = spark.read.format("xml").option("rowTag", src_config.get("row_tag", "transaction")).load(file_path)
elif file_format == "PARQUET":
    raw_df = spark.read.parquet(file_path)
else:
    raise ValueError(f"Unsupported file format: {file_format}")

print(f"[INFO] Loaded {raw_df.count()} rows from ABFSS.")


# 3. DYNAMIC CASTING & TARGET SCHEMA MAPPING

columns_meta = selected_source["schema_definition"]["columns"]
select_columns = []

for col_meta in columns_meta:
    src_col = col_meta["name"]
    tgt_col = col_meta.get("target_column", src_col)
    d_type = col_meta["data_type"]

    if "DECIMAL" in d_type:
        expr = F.col(src_col).cast(d_type)
    elif d_type == "DATE":
        fmt = col_meta.get("format", "yyyy-MM-dd")
        expr = F.to_date(F.col(src_col), fmt)
    else:
        expr = F.col(src_col).cast("string")

    select_columns.append(expr.alias(tgt_col))

mapped_df = raw_df.select(*select_columns)

# 4. DATA QUALITY LAYERING & QUARANTINE SEGREGATION

quality_rules = selected_source.get("quality_rules", [])
name_map = {c["name"]: c.get("target_column", c["name"]) for c in columns_meta}

validated_df = mapped_df.withColumn("is_quarantined", F.lit(False)) \
                        .withColumn("quarantine_reason", F.lit(None).cast("string"))

for rule in quality_rules:
    rule_id = rule["rule_id"]
    src_col = rule["column"]
    tgt_col = name_map.get(src_col, src_col)
    check_type = rule["check_type"]

    if check_type == "GREATER_THAN":
        threshold = rule["threshold"]
        fail_condition = F.col(tgt_col) <= threshold
        reason_msg = f"{rule_id}: {tgt_col} failed threshold check (> {threshold})"
    elif check_type == "UNIQUE":
        window_spec = Window.partitionBy(tgt_col)
        validated_df = validated_df.withColumn("_dup_count", F.count("*").over(window_spec))
        fail_condition = F.col("_dup_count") > 1
        reason_msg = f"{rule_id}: Duplicate key detected on {tgt_col}"
    else:
        continue

    validated_df = validated_df.withColumn(
        "quarantine_reason",
        F.when(
            fail_condition,
            F.when(F.col("quarantine_reason").isNull(), F.lit(reason_msg))
             .otherwise(F.concat_ws(" | ", F.col("quarantine_reason"), F.lit(reason_msg)))
        ).otherwise(F.col("quarantine_reason"))
    ).withColumn(
        "is_quarantined",
        F.when(fail_condition, F.lit(True)).otherwise(F.col("is_quarantined"))
    )

    if "_dup_count" in validated_df.columns:
        validated_df = validated_df.drop("_dup_count")

valid_df = validated_df.filter(F.col("is_quarantined") == False).drop("is_quarantined", "quarantine_reason")
quarantine_df = validated_df.filter(F.col("is_quarantined") == True)


# 5. TARGET BRONZE DELTA STORAGE WRITER (CLOUD TARGET)

dest = selected_source["destination_config"]
db_name = dest["target_database"]
table_name = dest["target_table"]
load_type = metadata["delivery_metadata"].get("load_type", "APPEND")

bronze_storage_location = f"{BASE_ABFSS_URL}/bronze_zone/{table_name}"
quarantine_storage_location = f"{BASE_ABFSS_URL}/bronze_zone/{table_name}_quarantine"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")
bronze_delta_exists = True
try:
    dbutils.fs.ls(f"{bronze_storage_location}/_delta_log")
except Exception:
    bronze_delta_exists = False

table_exists = spark.sql(f"SHOW TABLES IN {db_name} LIKE '{table_name}'").count() > 0
if table_exists and not bronze_delta_exists:
    spark.sql(f"DROP TABLE {db_name}.{table_name}")

save_mode = "overwrite" if load_type == "OVERWRITE" or not bronze_delta_exists else "append"

print(f"[WRITE] Saving valid records to ABFSS: {bronze_storage_location}")
valid_df.write.format("delta") \
    .mode(save_mode) \
    .option("path", bronze_storage_location) \
    .saveAsTable(f"{db_name}.{table_name}")

if quarantine_df.limit(1).count() > 0:
    print(f"[WARN] Saving invalid rows to ABFSS quarantine path: {quarantine_storage_location}")
    quarantine_df.write.format("delta") \
        .mode("append") \
        .option("path", quarantine_storage_location) \
        .saveAsTable(f"{db_name}.{table_name}_quarantine")

print("[SUCCESS] Cloud metadata-driven ingestion execution complete.")
display(valid_df)
